<a href="https://colab.research.google.com/github/fadeeva/MLDL_plgrnd/blob/master/CV/courses_notes/stepik__object_detection/2_nms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch

from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [2]:
def iou(box1, box2):
    '''
        Ограничевающие рамки в формате (x1, y1, x2, y2).

        Parameters:
            box1: torch.Tensor, size(4, )
            box2: torch.Tensor, size(4, )
        Returns:
            iou: torch.Tensor (scalar)
    '''

    b1x1, b1y1, b1x2, b1y2 = box1
    b2x1, b2y1, b2x2, b2y2 = box2

    area1 = (b1x2 - b1x1)*(b1y2 - b1y1)
    area2 = (b2x2 - b2x1)*(b2y2 - b2y1)

    x_left = torch.max(b1x1, b2x1)
    y_top = torch.max(b1y1, b2y1)
    x_right = torch.min(b1x2, b2x2)
    y_bottom = torch.min(b1y2, b2y2)

    if x_right < x_left or y_bottom < y_top:
        return torch.tensor(0, dtype=torch.float)

    w = x_right - x_left
    h = y_bottom - y_top

    inter = w*h
    union = area1 + area2 - inter
    iou = inter / union

    return iou

In [3]:
def box_iou(boxes1, boxes2):
    '''
        Ограничевающие рамки в формате (x1, y1, x2, y2)

        Parameters:
            boxes1: torch.Tensor, size(N, 4)
            boxes2: torch.Tensor, size(M, 4)
        Returns:
            iou: torch.Tensor, size(N, M)
    '''

    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2]) # size (N, M, 2)
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:]) # size (N, M, 2)

    wh = (rb - lt).clamp(min=0) # size (N, M, 2)

    inter = wh[..., 0] * wh[..., 1] # size (N, M)
    union = area1[:, None] + area2 - inter # size (N, M)

    iou = inter / union  # size (N, M)

    return iou


In [4]:
def xywh2xyxy(inp):
    '''
        Преобразование формата ограниченных рамок. (x, y, w, h) -> (x, y, x, y)
        Parameters:
            inp: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, w, h).
        Returns:
            out: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, x, y).
    '''

    out = torch.empty_like(inp)

    xy = inp[..., :2] # координаты x и y центра ограничевающей рамки
    wh = inp[..., 2:] / 2 # половина высоты и ширины ограничивающей рамки

    out[..., :2] = xy - wh # координаты x и y левого верхнего угла
    out[..., 2:] = xy + wh # координаты x и y правого нижнего угла

    return out


In [5]:
def xyxy2xywh(inp):
    '''
        Преобразование формата ограниченных рамок. (x, y, x, y) -> (x, y, w, h)
        Parameters:
            inp: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, x, y).
        Returns:
            out: torch.Tensor, size(batch_size, num_boxes, 4) or (num_boxes, 4).
                Формат ограниченных рамок (x, y, w, h).
    '''

    out = torch.empty_like(inp)

    # координаты центра ограничивающей рамки
    out[..., 0] = (inp[..., 0] + inp[..., 2]) / 2 # x
    out[..., 1] = (inp[..., 1] + inp[..., 3]) / 2 # y

    out[..., 2] = inp[..., 2] - inp[..., 0] # ширина рамки
    out[..., 3] = inp[..., 3] - inp[..., 1] # высота рамки

    return out


In [6]:
def nms(boxes, scores, threshold=.5):
    _, sorted_idx = scores.sort(descending=True)

    keep = []
    while sorted_idx.numel() > 0:
        if sorted_idx.numel() == 1:
            keep.append(sorted_idx)
            break

        idx = sorted_idx[0]
        keep.append(idx)

        boxs1 = boxes[sorted_idx[0]].unsqueeze(dim=0) # size (1, 4)
        boxs2 = boxes[sorted_idx[1:]]                 # size (M, 4)
        iou = box_iou(boxs1, boxs2)

        i = (iou < threshold).nonzero()[:, 1]
        if i.numel()==0:
            break

        sorted_idx = sorted_idx[i+1]

    return torch.tensor(keep, dtype=torch.int)


In [7]:
def non_max_suppression(
        pred,
        score_treshold=.25,
        iou_threshold=.45,
        agnostic=False,
        max_wh=7600,
        classes=None):

    '''
        Parameters:
            pred: torch.Tensor, size(batch-size, 4 + num_classes, num_boxes)/
                  Параметры ограничивающих рамок в формате (x, y, width, height).
            score_treshold: float(by default is .25), in interval [0, 1].
            iou_threshold: float(by default is .45), in interval [0, 1]/
            agnostic: bool (default is False), если False, то NMS считается
                      с учетом классов, если True, то NMS считается без учета
                      классов.
            max_wh: int (default is 7600), максимально возможная ширина и высота
                    входного изображения.
            classes: List[int] (default is None), список с индексами классоа,
                     которые нужно учитывать. Если None, то учитываются все
                     классы.
        Returns:
            output: List[torch.Tensor], список, длина которого batch_size,
                    содержащий тензоры с результатоми преобразований для каждого
                    элемента батча. Размер тензора (num_boxes, 4 + score + class)
    '''
    # bs - batch_size
    # nc - num_class
    # nb - num_boxes

    if classes is not None:
        classes = torch.tensor(classes, device=pred.device)

    bs = pred.shape[0]
    nc = pred.shape[1] - 4
    candidates = pred[:, 4:].amax(dim=1) > score_treshold # (bs, nb)

    pred = pred.transpose(-1, -2) # (bs, 4_nc, nb) -> (bs, nb, 4_nc)
    pred = torch.cat(xywh2xyxy(pred[..., :4]), pred[..., 4:], dim=-1) # xywh -> xyxy

    output = [torch.zeros((0, 6), device=pred.device)]*bs
    # Преобразуем предсказания для каждого изображения в батче
    for idx, pr in enumerate(pred):
        # idx - индекс изображения в батче
        # pr - предсказания для изображения. Size (nb, 4+nc)

        # Отбираем только те предсказания, к которых уверенность
        # в предсказанном классе (score) больше порогового (score_threshold)
        pr = pr[candidates[idx]]

        # Если нет подходящих предсказания, переходим к следующему изображению.
        if not pr.shape[0]:
            continue

        # Разделяем предсказания для одного изображения на предсказанные
        # ограничивающие рамки и предсказания классов.
        # box - предсказанные ограничивающие рамки, size (nb, nc)
        box, cls = pr.split((4, nc), dim=1)

        # Получаем индекс предсказанного класса и предсказанное значение (score).
        # При этом сохраняем размерность тензоров (с помощью аргумента keepdim).
        score, idx_cls = cls.max(dim=1, keepdim=True)

        # Расширенный тензор с предскзаниями, size (nb, 4 + score + idx_cls)
        pr = torch.cat((box, score, idx_cls.float()), dim=1)

        # Если указан аргумент classes, то в тензоре pr оставляем только
        # указанные классы.
        if classes is not None:
            i = (pr[:, 5:6] == classes).any(dim=1)
            pr = pr[i]

        # Если нет подходящих предсказаний, переходим к следующему изображению.
        if not pr.shape[0]:
            continue

        # Non Maximum Suppression.
        scores = pr[:, 4]
        scaling_for_classes = pr[:, 5:6] * (0 if agnostic else max_wh)
        boxes = pr[:, :4] + scaling_for_classes

        idx_nms = nms(boxes, scores, iou_threshold)

        output[idx] = pr[idx_nms]

    return output